In [ ]:
"""
CER – Demand Response Module and Load Optimization
===================================================
Bachelor's Thesis – Computer Engineering
Google Colab

Objective:
  Use ML forecasts to advise users to shift flexible loads
  to PV surplus hours, and simulate the impact on shared energy
  with full user participation (100% adoption).

Pipeline:
  1. Load ML forecasts (cer_previsioni.csv)
  2. Identify predicted PV surplus hours in the CER
  3. Detect flexible loads per user from historical profiles
  4. Generate recommendations: time slot + appliance
  5. Simulate load shifting (100% adoption – all users)
  6. Compare shared energy BEFORE and AFTER
  7. Calculate GSE incentive increase

NOTE: upload cer_previsioni.csv to Colab before running.
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
from google.colab import files

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.facecolor":"white","axes.facecolor":"white",
                     "axes.grid":True,"grid.alpha":0.3,"font.size":11})

COLORS = {"reale":"#185FA5","previsto":"#D85A30","fv":"#BA7517",
          "shared":"#0F6E56","dr":"#534AB7","neutral":"#888780"}
MESI   = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]

TARIFFA_GSE  = 0.130   # €/kWh – GSE shared energy incentive rate
ADESIONE     = 1.00    # 100% user adoption rate

print("  CER – Demand Response Simulation")
print("  Adoption: 100% of users · GSE rate: 130 €/MWh")


# ══════════════════════════════════════════════════════════════════════════════
# 1. DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════

print("\n[1/5] Loading forecasts ...")
df = pd.read_csv("cer_previsioni.csv", parse_dates=["timestamp"])
df = df.sort_values(["timestamp","member_id"]).reset_index(drop=True)

# Energy flows per user (individual self-consumption already in columns)
df["energia_immessa_kwh"] = pd.to_numeric(
    df.get("energia_immessa_kwh", 0), errors="coerce").fillna(0)
df["consumo_residuo_kwh"] = pd.to_numeric(
    df.get("consumo_residuo_kwh", 0), errors="coerce").fillna(0)

# Recalculate flows if columns are missing or zero
if df["energia_immessa_kwh"].sum() == 0:
    df["energia_immessa_kwh"] = np.maximum(
        df["produzione_fv_kwh"] - df["consumo_previsto_kwh"], 0)
    df["consumo_residuo_kwh"] = np.maximum(
        df["consumo_previsto_kwh"] - df["produzione_fv_kwh"], 0)

print(f"  Users in test set:    {df['member_id'].nunique()}")
print(f"  Months present:       {sorted(df['mese'].unique().tolist())}")
print(f"  Hours per user (M01): {df[df['member_id']=='M01'].shape[0]}")


# ══════════════════════════════════════════════════════════════════════════════
# 2. HOURLY PV SURPLUS AT CER LEVEL
# ══════════════════════════════════════════════════════════════════════════════

print("\n[2/5] Computing PV surplus ...")

# CER aggregation per hour
cer = df.groupby("timestamp").agg(
    consumo_reale_kwh       = ("consumo_kwh",           "sum"),
    consumo_previsto_kwh    = ("consumo_previsto_kwh",   "sum"),
    produzione_fv_kwh       = ("produzione_fv_kwh",      "sum"),
    energia_disponibile_kwh = ("energia_immessa_kwh",    "sum"),
    consumo_residuo_kwh     = ("consumo_residuo_kwh",    "sum"),
).reset_index()

cer["mese"] = cer["timestamp"].dt.month
cer["ora"]  = cer["timestamp"].dt.hour
cer["dow"]  = cer["timestamp"].dt.dayofweek

# Shared energy BEFORE Demand Response
cer["ec_prima_kwh"] = np.minimum(
    cer["consumo_residuo_kwh"],
    cer["energia_disponibile_kwh"],
)

# Surplus: available PV energy exceeding residual demand
cer["surplus_kwh"] = np.maximum(
    cer["energia_disponibile_kwh"] - cer["consumo_residuo_kwh"], 0)

tot_surplus = cer["surplus_kwh"].sum()
ore_surplus = (cer["surplus_kwh"] > 0.1).sum()
print(f"  Total PV surplus (test months): {tot_surplus:.0f} kWh")
print(f"  Hours with surplus > 0.1 kWh:  {ore_surplus}")
print(f"  Avg surplus per active hour:    {tot_surplus/ore_surplus:.2f} kWh")


# ══════════════════════════════════════════════════════════════════════════════
# 3. FLEXIBLE APPLIANCE CATALOGUE
#    Typical power draw and duration for each shiftable load
# ══════════════════════════════════════════════════════════════════════════════

ELETTRODOMESTICI = [
    # name,                    consumption_kwh, duration_h, compatible_user_types
    ("Washing machine",          1.80, 2, ["prosumer_res","consumer_res"]),
    ("Dishwasher",               1.20, 2, ["prosumer_res","consumer_res"]),
    ("Electric oven",            2.00, 1, ["prosumer_res","consumer_res"]),
    ("Tumble dryer",             2.50, 2, ["prosumer_res","consumer_res"]),
    ("EV charging",              3.50, 3, ["prosumer_res","consumer_res"]),
    ("Office printers/PCs",      0.80, 2, ["commercial"]),
    ("Air conditioning system",  3.00, 3, ["commercial"]),
    ("Device charging",          0.30, 2, ["prosumer_res","consumer_res","commercial"]),
]

def trova_elettrodomestico(surplus_disponibile_kwh, tipo_utente):
    """
    Given available surplus power and user type,
    return the list of recommended shiftable loads.
    """
    consigliati = []
    rimasto = surplus_disponibile_kwh
    for nome, consumo, durata, tipi in ELETTRODOMESTICI:
        if tipo_utente in tipi and consumo <= rimasto:
            consigliati.append((nome, consumo, durata))
            rimasto -= consumo
            if rimasto < 0.2:
                break
    return consigliati


# ══════════════════════════════════════════════════════════════════════════════
# 4. GENERATE PER-USER RECOMMENDATIONS
#    For each day in the test set, identify surplus hours
#    and assign flexible loads to shift.
# ══════════════════════════════════════════════════════════════════════════════

print("\n[3/5] Generating recommendations ...")

n_utenti = df["member_id"].nunique()
raccomandazioni = []
giorni = df["timestamp"].dt.normalize().unique()

for giorno in giorni:
    cer_day  = cer[cer["timestamp"].dt.normalize() == giorno].copy()
    ore_surp = cer_day[cer_day["surplus_kwh"] > 0.5].copy()

    if len(ore_surp) == 0:
        continue

    for _, member_row in df[df["timestamp"].dt.normalize()==giorno]\
                          .groupby("member_id").first().iterrows():
        mid  = member_row.name if hasattr(member_row,"name") else _
        tipo = member_row["tipo"]

        surplus_quota = ore_surp["surplus_kwh"].sum() / n_utenti

        if surplus_quota < 0.3:
            continue

        # Get user index (0–7) to stagger recommendations by 1 hour,
        # avoiding simultaneous load spikes across all users
        idx_utente    = list(df["member_id"].unique()).index(mid)
        ore_surp_list = sorted(ore_surp[ore_surp["surplus_kwh"] > 0.1]["ora"].tolist())

        if len(ore_surp_list) == 0:
            continue

        n_fasce   = max(len(ore_surp_list) - 1, 1)
        offset    = idx_utente % n_fasce
        ora_picco = ore_surp_list[min(offset, len(ore_surp_list) - 1)]
        ora_fine  = min(ora_picco + 2, 23)

        # Surplus available at the slot assigned to this user
        ora_surp_utente = ore_surp[ore_surp["ora"] == ora_picco]
        surplus_peak = (float(ora_surp_utente["surplus_kwh"].iloc[0])
                        if len(ora_surp_utente) > 0
                        else float(ore_surp["surplus_kwh"].mean()))

        carichi = trova_elettrodomestico(
            min(surplus_quota, surplus_peak * 0.8), tipo)

        if not carichi:
            continue

        carichi_str = " + ".join([f"{n} ({c:.1f} kWh)" for n,c,d in carichi])
        raccomandazioni.append({
            "data":        giorno,
            "member_id":   mid,
            "tipo":        tipo,
            "ora_inizio":  ora_picco,
            "ora_fine":    ora_fine,
            "surplus_kwh": round(surplus_quota, 2),
            "carichi":     carichi_str,
            "kwh_shift":   round(sum(c for _,c,_ in carichi), 2),
        })

df_rac = pd.DataFrame(raccomandazioni)
print(f"  Recommendations generated: {len(df_rac):,}")
print(f"  Users involved:            {df_rac['member_id'].nunique()}")
print(f"  Days with surplus:         {df_rac['data'].nunique()}")
print(f"  Total kWh to shift:        {df_rac['kwh_shift'].sum():.1f} kWh")

# Sample notification messages
print(f"\n  Sample user notifications")
for _, r in df_rac.head(6).iterrows():
    print(f"  [{r['member_id']}] {str(r['data'])[:10]}  "
          f"Slot {r['ora_inizio']:02d}:00–{r['ora_fine']:02d}:00  "
          f"→  {r['carichi']}")


# ══════════════════════════════════════════════════════════════════════════════
# 5. DEMAND RESPONSE SIMULATION
#
#    Simplified model: loads are shifted FROM the evening peak (19–21)
#    TO the recommended PV surplus slot. Total energy per user is
#    conserved – only the hourly distribution changes.
# ══════════════════════════════════════════════════════════════════════════════

print("\n[4/5] Simulating load shifting (100% adoption – all users) ...")

utenti_list = df["member_id"].unique().tolist()

# Build modification dict: {(timestamp, member_id): delta_kwh}
modifiche = {}

for _, rac in df_rac.iterrows():
    mid       = rac["member_id"]
    giorno    = pd.Timestamp(rac["data"])
    kwh_shift = rac["kwh_shift"]

    # Add load during recommended surplus hours
    ore_target  = list(range(rac["ora_inizio"], rac["ora_fine"] + 1))
    kwh_per_ora = kwh_shift / len(ore_target)
    for ora in ore_target:
        ts  = giorno + pd.Timedelta(hours=int(ora))
        key = (ts, mid)
        modifiche[key] = modifiche.get(key, 0) + kwh_per_ora

    # Reduce load during evening peak hours (19–21)
    ore_picco_serale = [19, 20, 21]
    kwh_riduzione    = kwh_shift / len(ore_picco_serale)
    for ora in ore_picco_serale:
        ts  = giorno + pd.Timedelta(hours=ora)
        key = (ts, mid)
        modifiche[key] = modifiche.get(key, 0) - kwh_riduzione

print(f"  Modifications applied: {len(modifiche):,} hourly slots")
n_aderenti = len(set(mid for _, mid in modifiche.keys()))
print(f"  Users adopted DR:      {n_aderenti} / {len(utenti_list)} (100%)")

# Apply modifications
df_dr = df.copy()
df_dr["consumo_dr_kwh"] = df_dr["consumo_previsto_kwh"].copy()

for (ts, mid), delta in modifiche.items():
    mask = (df_dr["timestamp"] == ts) & (df_dr["member_id"] == mid)
    if mask.any():
        df_dr.loc[mask, "consumo_dr_kwh"] = np.maximum(
            df_dr.loc[mask, "consumo_dr_kwh"] + delta, 0)

# Recalculate energy flows with modified consumption
df_dr["ei_dr"] = np.maximum(df_dr["produzione_fv_kwh"] - df_dr["consumo_dr_kwh"], 0)
df_dr["cr_dr"] = np.maximum(df_dr["consumo_dr_kwh"] - df_dr["produzione_fv_kwh"], 0)

# CER aggregation after DR
cer_dr = df_dr.groupby("timestamp").agg(
    consumo_dr_kwh         = ("consumo_dr_kwh", "sum"),
    energia_disponibile_dr = ("ei_dr",          "sum"),
    consumo_residuo_dr     = ("cr_dr",           "sum"),
).reset_index()

cer_dr["mese"] = cer_dr["timestamp"].dt.month
cer_dr["ora"]  = cer_dr["timestamp"].dt.hour

cer_merged = cer.merge(cer_dr, on="timestamp", suffixes=("","_dr"))
cer_merged["ec_dopo_kwh"] = np.minimum(
    cer_merged["consumo_residuo_dr"],
    cer_merged["energia_disponibile_dr"],
)


# ══════════════════════════════════════════════════════════════════════════════
# 6. QUANTITATIVE RESULTS
# ══════════════════════════════════════════════════════════════════════════════

ec_prima  = cer_merged["ec_prima_kwh"].sum()
ec_dopo   = cer_merged["ec_dopo_kwh"].sum()
delta_ec  = ec_dopo - ec_prima
delta_eur = delta_ec * TARIFFA_GSE

print(f"\n Demand Response Simulation Results")
print(f"  Shared energy BEFORE DR:    {ec_prima:>10.2f} kWh")
print(f"  Shared energy AFTER DR:     {ec_dopo:>10.2f} kWh")
print(f"  Absolute increase:          {delta_ec:>10.2f} kWh  "
      f"(+{100*delta_ec/ec_prima:.1f}%)")
print(f"  GSE incentive increase:     {delta_eur:>10.2f} €  "
      f"(rate: 130 €/MWh)")

print(f"\n Monthly breakdown")
per_mese = cer_merged.groupby("mese").agg(
    ec_prima = ("ec_prima_kwh", "sum"),
    ec_dopo  = ("ec_dopo_kwh",  "sum"),
).reset_index()
per_mese["delta_kwh"] = per_mese["ec_dopo"] - per_mese["ec_prima"]
per_mese["delta_pct"] = 100 * per_mese["delta_kwh"] / per_mese["ec_prima"]
per_mese["delta_eur"] = per_mese["delta_kwh"] * TARIFFA_GSE

print(f"  {'Month':<6} {'EC before':>10} {'EC after':>10} "
      f"{'Δ kWh':>8} {'Δ %':>6} {'Δ €':>7}")
print(f"  {'-'*50}")
for _, r in per_mese.iterrows():
    print(f"  {MESI[int(r['mese'])-1]:<6} "
          f"{r['ec_prima']:>10.1f} {r['ec_dopo']:>10.1f} "
          f"{r['delta_kwh']:>8.1f} {r['delta_pct']:>5.1f}% "
          f"{r['delta_eur']:>6.2f} €")
print(f"  {'TOTAL':<6} {per_mese['ec_prima'].sum():>10.1f} "
      f"{per_mese['ec_dopo'].sum():>10.1f} "
      f"{per_mese['delta_kwh'].sum():>8.1f}  "
      f"{delta_eur:>6.2f} €")


# ══════════════════════════════════════════════════════════════════════════════
# 7. CHARTS
# ══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("I  –  Demand Response · load shifting impact on the CER",
             fontsize=13, fontweight="bold")

# ── I-1: Average hourly CER consumption profile before/after DR ──────────────
ax = axes[0]
profilo_prima = cer_merged.groupby("ora")["consumo_previsto_kwh"].mean()
profilo_dopo  = cer_dr.groupby("ora")["consumo_dr_kwh"].mean()
fv_medio      = cer_merged.groupby("ora")["produzione_fv_kwh"].mean()

ax.fill_between(range(24), fv_medio, alpha=0.20,
                color=COLORS["fv"], label="Average PV production")
ax.plot(range(24), profilo_prima, color=COLORS["reale"],
        lw=2, label="CER consumption (baseline)")
ax.plot(range(24), profilo_dopo,  color=COLORS["dr"],
        lw=2, ls="--", label="CER consumption (after DR)")
ax.fill_between(range(24), profilo_prima, profilo_dopo,
                where=(profilo_dopo > profilo_prima),
                alpha=0.3, color=COLORS["dr"], label="Added loads")
ax.fill_between(range(24), profilo_prima, profilo_dopo,
                where=(profilo_dopo < profilo_prima),
                alpha=0.3, color=COLORS["shared"], label="Reduced loads")
ax.set_xlabel("Hour of day")
ax.set_ylabel("kWh/hour (CER average)")
ax.set_title("Average hourly consumption · before vs after DR")
ax.set_xticks(range(0, 24, 2))
ax.legend(fontsize=8)

# ── I-2: Shared energy per month before/after DR ─────────────────────────────
ax = axes[1]
x  = np.arange(len(per_mese))
w  = 0.35
ax.bar(x - w/2, per_mese["ec_prima"], w,
       color=COLORS["reale"], alpha=0.85,
       edgecolor="white", label="Shared energy (baseline)")
ax.bar(x + w/2, per_mese["ec_dopo"], w,
       color=COLORS["dr"], alpha=0.85,
       edgecolor="white", label="Shared energy (after DR)")
for i, (p, d) in enumerate(zip(per_mese["ec_prima"], per_mese["ec_dopo"])):
    if d > p:
        ax.annotate(f"+{d-p:.0f}\nkWh",
                    xy=(i + w/2, d), xytext=(i + w/2, d + 5),
                    ha="center", fontsize=8,
                    color=COLORS["dr"], fontweight="bold")
ax.set_xticks(x)
ax.set_xticklabels([MESI[m-1] for m in per_mese["mese"]])
ax.set_ylabel("kWh")
ax.set_title("Shared energy per month · before vs after DR")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig("grafico_I_demand_response.png", dpi=150, bbox_inches="tight")
plt.show()


# ══════════════════════════════════════════════════════════════════════════════
# 8. SAVE OUTPUT AND DOWNLOAD
# ══════════════════════════════════════════════════════════════════════════════

df_rac.to_csv("cer_raccomandazioni_dr.csv", index=False)

import zipfile, os, base64
from IPython.display import display, HTML

file_list = ["cer_raccomandazioni_dr.csv", "grafico_I_demand_response.png"]

with zipfile.ZipFile("CER_demand_response_100.zip", "w") as z:
    for f in file_list:
        if os.path.exists(f):
            z.write(f); print(f"  + {f}")

with open("CER_demand_response_100.zip", "rb") as f:
    b64 = base64.b64encode(f.read()).decode()

display(HTML(f"""
<a download="CER_demand_response_100.zip"
   href="data:application/zip;base64,{b64}"
   style="display:inline-block;padding:12px 24px;background:#534AB7;
          color:white;border-radius:8px;text-decoration:none;
          font-size:14px;font-weight:500;">
  Download CER_demand_response_100.zip
</a>
"""))
print("\n✓ Click the button to download the results.")